# Tragedy of the Commons — Interactive Simulation

**ECON7720: Ecological & Environmental Economics** | Lecture 02 | The University of Queensland

---

## Model

$n$ identical herders share a pasture. Each herder $i$ chooses $a_i \geq 0$ animals.  
Total animals: $A = \sum_{j=1}^{n} a_j$.  
Value per animal falls with congestion: $v(A) = 10 - A$.  
Cost per animal: $c = 2$ (constant).

### Three regimes

| Regime | FOC | Solution | Rent |
|---|---|---|---|
| **Social planner** | $v(A) + A\,v'(A) = c$ | $A^* = 4$ | $16$ |
| **Nash equilibrium** ($n$ herders) | $v(A) + \tfrac{A}{n}\,v'(A) = c$ | $A^N = \dfrac{8n}{n+1}$ | $\dfrac{64n}{(n+1)^2}$ |
| **Open access** ($n \to \infty$) | $v(A) = c$ | $A_{OA} = 8$ | $0$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from IPython.display import display, Markdown

In [ ]:
# ── Model parameters ──────────────────────────────────────────────
V_MAX = 10      # v(0)
C = 2           # cost per animal

# Closed-form solutions
A_STAR = (V_MAX - C) / 2          # = 4
A_OA   = V_MAX - C                # = 8
RENT_STAR = A_STAR * (V_MAX - A_STAR - C)  # = 16

def v(A):
    return V_MAX - A

def total_rent(A):
    return A * (v(A) - C)

def nash_A(n):
    return (V_MAX - C) * n / (n + 1)

def nash_rent(n):
    return (V_MAX - C)**2 * n / (n + 1)**2

# UQ colours
UQ_PURPLE = "#512478"
UQ_CYAN   = "#0099CC"

## 1. Summary table

How the Nash equilibrium changes as the number of herders $n$ grows:

In [ ]:
print(f"{'n':>5s}  {'A^N':>6s}  {'Rent':>7s}  {'% lost':>7s}")
print("-" * 30)
for n in [1, 2, 3, 5, 10, 20, 50, 100]:
    An = nash_A(n)
    Rn = nash_rent(n)
    pct = (1 - Rn / RENT_STAR) * 100
    print(f"{n:5d}  {An:6.2f}  {Rn:7.2f}  {pct:6.1f}%")
print(f"{'∞':>5s}  {A_OA:6.2f}  {'0.00':>7s}  {'100.0%':>7s}")

## 2. Interactive simulation

Use the slider to change the number of herders and watch:
- **Left:** where the Nash equilibrium sits on the rent curve
- **Centre:** convergence of $A^N$ and rent as $n$ grows
- **Right:** individual herder's best-response payoff

In [ ]:
def simulate(n=5):
    An = nash_A(n)
    Rn = nash_rent(n)
    pct_lost = (1 - Rn / RENT_STAR) * 100

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    fig.suptitle(f"Tragedy of the Commons — n = {n} herders",
                 fontsize=13, fontweight="bold", color=UQ_PURPLE)

    # ── Panel 1: Rent curve ──────────────────────────────────────
    ax1 = axes[0]
    A_grid = np.linspace(0, 10, 300)
    ax1.plot(A_grid, total_rent(A_grid), color=UQ_PURPLE, lw=2,
             label="Rent $= A(v(A)-c)$")
    ax1.axhline(0, color="grey", lw=0.5)
    ax1.axvline(A_STAR, color="grey", ls="--", lw=1,
                label=f"$A^*={A_STAR:.0f}$")
    ax1.axvline(A_OA, color=UQ_CYAN, ls="--", lw=1,
                label=f"$A_{{OA}}={A_OA:.0f}$")
    ax1.plot(An, Rn, "o", color=UQ_CYAN, ms=10, zorder=5)
    ax1.annotate(f"$n={n}$: $A^N={An:.2f}$\nRent={Rn:.2f} ({pct_lost:.0f}% lost)",
                 xy=(An, Rn), xytext=(An + 0.4, Rn + 3),
                 fontsize=9, color=UQ_CYAN,
                 arrowprops=dict(arrowstyle="->", color=UQ_CYAN))
    ax1.set_xlabel("Total animals $A$")
    ax1.set_ylabel("Rent ($)")
    ax1.set_title("Rent curve", color=UQ_PURPLE, fontsize=11)
    ax1.legend(fontsize=8, loc="upper right")
    ax1.set_xlim(0, 10)
    ax1.set_ylim(-6, 20)

    # ── Panel 2: Convergence ─────────────────────────────────────
    ax2 = axes[1]
    n_grid = np.arange(1, 101)
    ax2.plot(n_grid, nash_A(n_grid), color=UQ_PURPLE, lw=2, label="$A^N$")
    ax2.axhline(A_STAR, color="grey", ls="--", lw=1)
    ax2.axhline(A_OA, color=UQ_CYAN, ls="--", lw=1)
    ax2.text(102, A_STAR, "$A^*$", va="center", fontsize=9, color="grey")
    ax2.text(102, A_OA, "$A_{OA}$", va="center", fontsize=9, color=UQ_CYAN)
    ax2.plot(n, An, "o", color=UQ_PURPLE, ms=10, zorder=5)

    ax2b = ax2.twinx()
    ax2b.plot(n_grid, nash_rent(n_grid), color=UQ_CYAN, lw=2, label="Rent")
    ax2b.axhline(RENT_STAR, color="grey", ls=":", lw=1)
    ax2b.text(102, RENT_STAR, f"Rent$^*$={RENT_STAR:.0f}", va="center",
              fontsize=8, color="grey")
    ax2b.plot(n, Rn, "o", color=UQ_CYAN, ms=10, zorder=5)

    ax2.set_xlabel("Number of herders $n$")
    ax2.set_ylabel("$A^N$", color=UQ_PURPLE)
    ax2b.set_ylabel("Rent ($)", color=UQ_CYAN)
    ax2.set_title("Convergence to open access", color=UQ_PURPLE, fontsize=11)
    ax2.set_xlim(1, 100)
    ax2.set_ylim(3, 9)
    ax2b.set_ylim(0, 18)

    # ── Panel 3: Best response ───────────────────────────────────
    ax3 = axes[2]
    ai_grid = np.linspace(0, max(5, An / n * 2.5), 200)
    # Others play Nash
    a_others = An * (n - 1) / n if n > 1 else 0
    payoffs = ai_grid * (v(ai_grid + a_others) - C)

    ax3.plot(ai_grid, payoffs, color=UQ_PURPLE, lw=2, label="$\\pi_i(a_i)$")
    ax3.axhline(0, color="grey", lw=0.5)

    # Nash choice
    ai_nash = An / n
    pi_nash = ai_nash * (v(ai_nash + a_others) - C)
    ax3.plot(ai_nash, pi_nash, "o", color=UQ_CYAN, ms=10, zorder=5,
             label=f"Nash $a_i^N={ai_nash:.2f}$")

    # Planner choice
    ai_planner = A_STAR / n
    pi_planner = ai_planner * (v(ai_planner + a_others) - C)
    ax3.plot(ai_planner, pi_planner, "s", color="grey", ms=8, zorder=5,
             label=f"Planner $a_i^*={ai_planner:.2f}$")

    ax3.set_xlabel("Herder $i$'s animals $a_i$")
    ax3.set_ylabel("Herder $i$'s profit $\\pi_i$ ($)")
    ax3.set_title("Individual best response", color=UQ_PURPLE, fontsize=11)
    ax3.legend(fontsize=8, loc="upper right")

    plt.tight_layout()
    plt.show()

interact(simulate, n=IntSlider(min=1, max=100, step=1, value=5,
                               description="Herders n:",
                               style={"description_width": "80px"},
                               layout={"width": "400px"}));

## 3. Key takeaways

1. **The tragedy is continuous, not binary.** With just 5 herders, 44% of the rent is already gone.
2. **The wedge grows with $n$.** Each herder internalises only $1/n$ of the congestion externality.
3. **Nash $\to$ open access as $n \to \infty$.** The $1/n$ term vanishes and rent is fully dissipated.
4. **The best-response panel shows why no one deviates.** At the Nash equilibrium, each herder is individually optimising — the problem is that the *collective* outcome is inefficient.

---

### Exercises

Try modifying the model parameters in the code above:
- What happens if the cost $c$ rises (e.g. $c = 5$)? Does the tragedy shrink?
- What if $v(A) = 10 - A^2$ (sharper congestion)? Does rent dissipate faster or slower?
- A **Pigouvian tax** of $t$ per animal raises effective cost to $c + t$. What $t$ restores efficiency?

---
*ECON7720 — Dr Juan Soto-Diaz, School of Economics & SMI, The University of Queensland*